# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [3]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
openai = OpenAI(base_url=os.getenv('API_BASE_URL'))

In [5]:
# Some lists!

todos = []
completed = []

In [6]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [7]:
get_todo_report()

''

In [8]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [10]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [14]:
completed

[True, False, False]

In [15]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [16]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [17]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [18]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5", messages=messages, tools=tools, reasoning_effort="low")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [26]:
todos, completed = [], []
loop(messages)

Todo #1: Estimate the distance between Boston and New York to replace the missing value.
Todo #2: Compute how far the Boston train travels from 2:00–3:00 pm.
Todo #3: Compute the remaining distance between trains at 3:00 pm.
Todo #4: Compute their closing speed after 3:00 pm.
Todo #5: Compute the time after 3:00 pm until they meet.
Todo #6: Convert that time to a clock time and give the meeting time.
Todo #7: Optionally, note how the answer changes if a different common Boston–NYC distance is used.

Common driving/rail route distance Boston–NYC is about 215–220 miles; straight-line is ~190 miles. For intercity 
trains, route distance is roughly 230 miles by Amtrak Northeast Corridor including Providence–New Haven corridor. 
I'll select 215 miles as a reasonable canonical estimate for back-of-envelope.

Todo #1: Estimate the distance between Boston and New York to replace the missing value.
Todo #2: Compute how far the Boston train travels from 2:00–3:00 pm.
Todo #3: Compute the remaining distance between trains at 3:00 pm.
Todo #4: Compute their closing speed after 3:00 pm.
Todo #5: Compute the time after 3:00 pm until they meet.
Todo #6: Convert that time to a clock time and give the meeting time.
Todo #7: Optionally, note how the answer changes if a different common Boston–NYC distance is used.

From 2:00 to 3:00 pm is 1 hour. At 60 mph, distance = 60 miles.

Todo #1: Estimate the distance between Boston and New York to replace the missing value.
Todo #2: Compute how far the Boston train travels from 2:00–3:00 pm.
Todo #3: Compute the remaining distance between trains at 3:00 pm.
Todo #4: Compute their closing speed after 3:00 pm.
Todo #5: Compute the time after 3:00 pm until they meet.
Todo #6: Convert that time to a clock time and give the meeting time.
Todo #7: Optionally, note how the answer changes if a different common Boston–NYC distance is used.

...

Add to 3:00 pm: 1 hr 6 min 26 sec gives approximately 4:06:26 pm meeting time.

Todo #1: Estimate the distance between Boston and New York to replace the missing value.
Todo #2: Compute how far the Boston train travels from 2:00–3:00 pm.
Todo #3: Compute the remaining distance between trains at 3:00 pm.
Todo #4: Compute their closing speed after 3:00 pm.
Todo #5: Compute the time after 3:00 pm until they meet.
Todo #6: Convert that time to a clock time and give the meeting time.
Todo #7: Optionally, note how the answer changes if a different common Boston–NYC distance is used.

If instead you assume 225 miles, remaining distance at 3 pm = 165 miles; time after 3 pm = 165/140 = 1.1786 hr ≈ 1 
hr 10.7 min, meeting ≈ 4:10–4:11 pm. If 200 miles total, remaining = 140; time = 1.0 hr, meeting at 4:00 pm. So 
answer varies a few minutes depending on the assumed distance.

Todo #1: Estimate the distance between Boston and New York to replace the missing value.
Todo #2: Compute how far the Boston train travels from 2:00–3:00 pm.
Todo #3: Compute the remaining distance between trains at 3:00 pm.
Todo #4: Compute their closing speed after 3:00 pm.
Todo #5: Compute the time after 3:00 pm until they meet.
Todo #6: Convert that time to a clock time and give the meeting time.
Todo #7: Optionally, note how the answer changes if a different common Boston–NYC distance is used.

Answer (with a reasonable distance estimate):
- Assumed Boston–New York distance: 215 miles (typical driving/rail estimate).
- By 3:00 pm, Boston train has gone 60 miles, leaving 155 miles between trains.
- After 3:00 pm they approach each other at 60 + 80 = 140 mph.
- Time to meet after 3:00 pm = 155 / 140 ≈ 1.107 hr ≈ 1 hr 6 min 26 sec.
- Meeting time ≈ 4:06 pm (about 4:06:26 pm).

Sensitivity to the assumed distance:
- If distance were 200 miles: meet at 4:00 pm.
- If distance were 225 miles: meet around 4:10–4:11 pm.

Thus, with a typical estimate, they meet at about 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>